Research Question 8:

Which narratives about food inflation appear most frequently across news articles, YouTube videos, and YouTube comments?

In [19]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
# File paths
FILES_DIR = "."   # same folder as the notebook

NEWS_FILE = os.path.join(FILES_DIR, "news_data.json")
VIDEOS_FILE = os.path.join(FILES_DIR, "youtube_food_inflation_germany_videos.json")
COMMENTS_FILE = os.path.join(FILES_DIR, "youtube_food_inflation_comments_germany.json")

# Analysis period
START_DATE = "2023-01-01"
END_DATE = "2026-12-31"


# Minimum text lengths for filtering
MIN_LENGTH_COMMENT = 5
MIN_LENGTH_TEXT = 5

In [21]:
def load_news_json(filepath):
    """
    Load news data from a JSON file and return it as a DataFrame.

    This function supports two possible structures:
    1. A dictionary containing an "articles" key
    2. A direct list of article objects

    It also removes the prefix "news api:" if it appears at the
    beginning of the file content.
    """
    # Open the file and read its full content as text.
    with open(filepath, "r", encoding="utf-8-sig") as file:
        content = file.read().strip()

    # Remove an optional text prefix before parsing the JSON.
    if content.lower().startswith("news api:"):
        content = content[len("news api:"):].strip()

    # Convert the JSON text into a Python object.
    raw_data = json.loads(content)

    # Case 1: the JSON is a dictionary with an "articles" field.
    if isinstance(raw_data, dict) and "articles" in raw_data:
        dataframe = pd.DataFrame(raw_data["articles"])
        return dataframe

    # Case 2: the JSON is already a list of articles.
    if isinstance(raw_data, list):
        dataframe = pd.DataFrame(raw_data)
        return dataframe

    # Raise an error if the JSON structure is not supported.
    raise ValueError("Unsupported structure in news_data.json")


def load_videos_json(filepath):
    """
    Load YouTube video data from a JSON file and return it as a DataFrame.
    """
    # Open the file and parse the JSON content directly.
    with open(filepath, "r", encoding="utf-8-sig") as file:
        raw_data = json.load(file)

    # Convert the loaded video data into a DataFrame.
    dataframe = pd.DataFrame(raw_data)
    return dataframe


def load_comments_json(filepath):
    """
    Load YouTube comment data from a JSON file and return it as a DataFrame.
    """
    # Open the file and parse the JSON content directly.
    with open(filepath, "r", encoding="utf-8-sig") as file:
        raw_data = json.load(file)

    # Store the comments in a DataFrame with one column named "comment".
    dataframe = pd.DataFrame(raw_data, columns=["comment"])
    return dataframe

In [22]:
def clean_news_data(dataframe):
    """
    Clean news article data.
    """
    # Create a copy so the original input DataFrame is not modified.
    cleaned = dataframe.copy()

    # Keep only the columns that are relevant for the analysis.
    useful_columns = ["title", "description", "publishedAt", "content"]
    available_columns = [
        column for column in useful_columns if column in cleaned.columns
    ]
    cleaned = cleaned[available_columns].copy()

    # Standardize text columns by converting them to strings and
    # removing leading and trailing whitespace.
    cleaned["title"] = cleaned["title"].astype(str).str.strip()
    cleaned["description"] = (
        cleaned["description"].fillna("").astype(str).str.strip()
    )
    cleaned["content"] = cleaned["content"].fillna("").astype(str).str.strip()

    # Convert the publication date to datetime format.
    # Invalid dates are turned into missing values.
    cleaned["publishedAt"] = pd.to_datetime(
        cleaned["publishedAt"],
        errors="coerce",
        utc=True,
    )

    # Remove rows with missing title or missing publication date.
    cleaned = cleaned.dropna(subset=["title", "publishedAt"])

    # Remove very short titles that are unlikely to be useful.
    cleaned = cleaned[cleaned["title"].str.len() >= MIN_LENGTH_TEXT]

    # Remove duplicate articles based on title and publication date.
    cleaned = cleaned.drop_duplicates(subset=["title", "publishedAt"])

    # Sort the dataset chronologically and reset the index.
    cleaned = cleaned.sort_values("publishedAt").reset_index(drop=True)

    # Keep only articles within the selected analysis period.
    cleaned = cleaned[
        (cleaned["publishedAt"] >= START_DATE)
        & (cleaned["publishedAt"] <= END_DATE)
    ].copy()

    return cleaned


def clean_videos_data(dataframe):
    """
    Clean YouTube video data.
    """
    # Create a copy so the original input DataFrame is not modified.
    cleaned = dataframe.copy()

    # Keep only the columns that are relevant for the analysis.
    useful_columns = [
        "video_id",
        "title",
        "description",
        "channel",
        "published_at",
    ]
    available_columns = [
        column for column in useful_columns if column in cleaned.columns
    ]
    cleaned = cleaned[available_columns].copy()

    # Standardize text fields by converting them to strings and
    # removing leading and trailing whitespace.
    cleaned["title"] = cleaned["title"].astype(str).str.strip()
    cleaned["description"] = (
        cleaned["description"].fillna("").astype(str).str.strip()
    )
    cleaned["channel"] = cleaned["channel"].fillna("").astype(str).str.strip()

    # Convert the upload date to datetime format.
    # Invalid dates are turned into missing values.
    cleaned["published_at"] = pd.to_datetime(
        cleaned["published_at"],
        errors="coerce",
        utc=True,
    )

    # Remove rows with missing title or missing publication date.
    cleaned = cleaned.dropna(subset=["title", "published_at"])

    # Remove very short titles that are unlikely to be informative.
    cleaned = cleaned[cleaned["title"].str.len() >= MIN_LENGTH_TEXT]

    # Remove duplicate videos based on the unique video ID.
    cleaned = cleaned.drop_duplicates(subset=["video_id"])

    # Sort the dataset chronologically and reset the index.
    cleaned = cleaned.sort_values("published_at").reset_index(drop=True)

    # Keep only videos within the selected analysis period.
    cleaned = cleaned[
        (cleaned["published_at"] >= START_DATE)
        & (cleaned["published_at"] <= END_DATE)
    ].copy()

    return cleaned


def clean_comments_data(dataframe):
    """
    Clean comment data.
    """
    # Create a copy so the original input DataFrame is not modified.
    cleaned = dataframe.copy()

    # Standardize the comment text by converting it to a string and
    # removing leading and trailing whitespace.
    cleaned["comment"] = cleaned["comment"].astype(str).str.strip()

    # Remove rows with missing comments.
    cleaned = cleaned.dropna(subset=["comment"])

    # Remove comments that are too short to be meaningful.
    cleaned = cleaned[cleaned["comment"].str.len() >= MIN_LENGTH_COMMENT]

    # Remove exact duplicate comments.
    cleaned = cleaned.drop_duplicates(subset=["comment"])

    # Reset the index after cleaning.
    cleaned = cleaned.reset_index(drop=True)

    return cleaned

In [23]:
def add_news_features(dataframe):
    """
    Add time variables and a combined text field to the news data.
    """
    # Create a copy so the original input DataFrame is not modified.
    result = dataframe.copy()

    # Remove timezone information to simplify further time-based analysis.
    result["publishedAt"] = result["publishedAt"].dt.tz_localize(None)

    # Extract year and month information from the publication date.
    result["year"] = result["publishedAt"].dt.year
    result["month_number"] = result["publishedAt"].dt.month

    # Create a normalized monthly timestamp for grouping by month.
    result["month"] = result["publishedAt"].dt.to_period("M").dt.to_timestamp()

    # Combine title, description, and content into one lowercase text field
    # so that each article can be analyzed as a single text unit.
    result["text"] = (
        result["title"].fillna("")
        + " "
        + result["description"].fillna("")
        + " "
        + result["content"].fillna("")
    ).str.lower()

    # Add a platform label to identify the source later in comparisons.
    result["platform"] = "news"

    return result


def add_video_features(dataframe):
    """
    Add time variables and a combined text field to the video data.
    """
    # Create a copy so the original input DataFrame is not modified.
    result = dataframe.copy()

    # Remove timezone information to simplify further time-based analysis.
    result["published_at"] = result["published_at"].dt.tz_localize(None)

    # Extract year and month information from the publication date.
    result["year"] = result["published_at"].dt.year
    result["month_number"] = result["published_at"].dt.month

    # Create a normalized monthly timestamp for grouping by month.
    result["month"] = result["published_at"].dt.to_period("M").dt.to_timestamp()

    # Combine title and description into one lowercase text field
    # so that each video can be analyzed as a single text unit.
    result["text"] = (
        result["title"].fillna("")
        + " "
        + result["description"].fillna("")
    ).str.lower()

    # Add a platform label to identify the source later in comparisons.
    result["platform"] = "youtube_videos"

    return result


def add_comment_features(dataframe):
    """
    Add a text field and a platform label to the comment data.
    """
    # Create a copy so the original input DataFrame is not modified.
    result = dataframe.copy()

    # Convert the comment text to lowercase for consistent text matching.
    result["text"] = result["comment"].fillna("").str.lower()

    # Add a platform label to identify the source later in comparisons.
    result["platform"] = "youtube_comments"

    return result

In [29]:
def assign_narrative(text):
    """
    Assign one narrative category using simple keyword rules.
    """
    # Convert the input to lowercase text to ensure consistent matching.
    text = str(text).lower()

    # Define keywords related to the narrative of corporate greed.
    corporate_greed_keywords = [
        "gier", "profit", "profite", "monopol", "abzocke", "abzock",
        "rekordgewinne", "konzerne", "margen", "supermarktketten",
        "handelsketten", "unternehmen"
    ]

    # Define keywords related to the narrative of political failure.
    political_failure_keywords = [
        "politik", "regierung", "ampel", "cdu", "spd", "grüne",
        "bundesregierung", "staat", "fehlpolitik", "minister", "parteien"
    ]

    # Define keywords related to monetary or economic explanations.
    monetary_causes_keywords = [
        "ezb", "geldmenge", "gelddruck", "zinsen", "währung",
        "euro", "kaufkraft", "leitzins", "geldpolitik", "inflation"
    ]

    # Define keywords related to energy, taxes, and production costs.
    energy_tax_costs_keywords = [
        "co2", "co2-steuer", "energiepreise", "gas", "strom",
        "transport", "dünger", "sanktionen", "steuer", "kosten"
    ]

    # Assign the first matching narrative based on keyword presence.
    # The order matters because each text receives only one label.
    if any(keyword in text for keyword in corporate_greed_keywords):
        return "corporate_greed"

    if any(keyword in text for keyword in political_failure_keywords):
        return "political_failure"

    if any(keyword in text for keyword in monetary_causes_keywords):
        return "monetary_causes"

    if any(keyword in text for keyword in energy_tax_costs_keywords):
        return "energy_tax_costs"

    # Assign "other" if no keyword rule matches.
    return "other"


def classify_narratives(dataframe, text_column="text"):
    """
    Assign a narrative label to each row of a DataFrame.
    """
    # Create a copy so the original input DataFrame is not modified.
    result = dataframe.copy()

    # Apply the rule-based narrative classification to the selected text column.
    result["narrative"] = result[text_column].apply(assign_narrative)

    return result

In [30]:
# Load the raw datasets from the three input files.
news_raw = load_news_json(NEWS_FILE)
videos_raw = load_videos_json(VIDEOS_FILE)
comments_raw = load_comments_json(COMMENTS_FILE)

# Print the size of each raw dataset as a quick sanity check.
print("News raw shape:", news_raw.shape)
print("Videos raw shape:", videos_raw.shape)
print("Comments raw shape:", comments_raw.shape)

News raw shape: (58, 8)
Videos raw shape: (50, 5)
Comments raw shape: (541, 1)


In [31]:
# Apply cleaning functions to each dataset to remove invalid,
# duplicate, or incomplete entries.
news_clean = clean_news_data(news_raw)
videos_clean = clean_videos_data(videos_raw)
comments_clean = clean_comments_data(comments_raw)

# Print the size of each cleaned dataset to see how many rows remain
# after preprocessing.
print("News cleaned shape:", news_clean.shape)
print("Videos cleaned shape:", videos_clean.shape)
print("Comments cleaned shape:", comments_clean.shape)

News cleaned shape: (53, 4)
Videos cleaned shape: (49, 5)
Comments cleaned shape: (539, 1)


In [33]:
# Add feature columns to each cleaned dataset.
# This includes combined text fields, time variables, and platform labels.
news_features = add_news_features(news_clean)
videos_features = add_video_features(videos_clean)
comments_features = add_comment_features(comments_clean)

# Classify each text into one narrative category using the rule-based
# keyword matching function.
news_classified = classify_narratives(news_features)
videos_classified = classify_narratives(videos_features)
comments_classified = classify_narratives(comments_features)

# Print the shape of each classified dataset to confirm that the
# classification step was completed successfully.
print("News classified shape:", news_classified.shape)
print("Videos classified shape:", videos_classified.shape)
print("Comments classified shape:", comments_classified.shape)

News classified shape: (53, 10)
Videos classified shape: (49, 11)
Comments classified shape: (539, 4)


In [34]:
# Count how often each narrative appears in the news dataset.
# The number of rows per narrative is stored in the column "count".
news_narrative_counts = (
    news_classified.groupby("narrative", as_index=False)
    .agg(count=("title", "count"))
)

# Add a platform label so the counts can later be compared
# across all data sources.
news_narrative_counts["platform"] = "news"

# Count how often each narrative appears in the YouTube video dataset.
videos_narrative_counts = (
    videos_classified.groupby("narrative", as_index=False)
    .agg(count=("video_id", "count"))
)

# Add a platform label for later comparison.
videos_narrative_counts["platform"] = "youtube_videos"

# Count how often each narrative appears in the YouTube comments dataset.
comments_narrative_counts = (
    comments_classified.groupby("narrative", as_index=False)
    .agg(count=("comment", "count"))
)

# Add a platform label for later comparison.
comments_narrative_counts["platform"] = "youtube_comments"

# Combine the three summary tables into one dataset so that
# all platforms can be analyzed and visualized together.
combined_counts = pd.concat(
    [
        news_narrative_counts,
        videos_narrative_counts,
        comments_narrative_counts,
    ],
    ignore_index=True,
)

# Display the combined count table.
combined_counts

,narrative,count,platform
0,corporate_greed,10,news
1,energy_tax_costs,1,news
2,monetary_causes,27,news
3,other,7,news
4,political_failure,8,news
5,corporate_greed,6,youtube_videos
6,energy_tax_costs,5,youtube_videos
7,monetary_causes,19,youtube_videos
8,other,15,youtube_videos
9,political_failure,4,youtube_videos


In [35]:
# Count how often each narrative appears per year in the news dataset.
# This allows us to analyze how narrative frequencies change over time.
news_yearly = (
    news_classified.groupby(["year", "narrative"], as_index=False)
    .agg(count=("title", "count"))
)

# Add a platform label so yearly news counts can later be compared
# with yearly video counts.
news_yearly["platform"] = "news"

# Count how often each narrative appears per year in the YouTube
# video dataset.
videos_yearly = (
    videos_classified.groupby(["year", "narrative"], as_index=False)
    .agg(count=("video_id", "count"))
)

# Add a platform label for later comparison.
videos_yearly["platform"] = "youtube_videos"

# Combine yearly counts from news and YouTube videos into one table.
# Comments are not included here because no yearly time variable
# was added to the comment dataset.
media_yearly = pd.concat(
    [news_yearly, videos_yearly],
    ignore_index=True,
)

# Display the first rows of the combined yearly summary table.
media_yearly.head()

,year,narrative,count,platform
0,2026,corporate_greed,10,news
1,2026,energy_tax_costs,1,news
2,2026,monetary_causes,27,news
3,2026,other,7,news
4,2026,political_failure,8,news


In [36]:
# Reshape the combined count table into a wide summary format.
# Each row represents one narrative, and each column represents
# one platform.
summary_table = combined_counts.pivot_table(
    index="narrative",
    columns="platform",
    values="count",
    aggfunc="sum",
    fill_value=0,
)

# Display the summary table for easier comparison across platforms.
summary_table

platform,news,youtube_comments,youtube_videos
narrative,,,
corporate_greed,10,51,6
energy_tax_costs,1,35,5
monetary_causes,27,57,19
other,7,336,15
political_failure,8,60,4


In [37]:
sns.set_theme(style="whitegrid")

In [40]:
# Identify the most frequent narrative in the news dataset.
print("Top narrative in news:")
print(news_narrative_counts.sort_values("count", ascending=False).head(1))
print()

# Identify the most frequent narrative in the YouTube video dataset.
print("Top narrative in YouTube videos:")
print(videos_narrative_counts.sort_values("count", ascending=False).head(1))
print()

# Identify the most frequent narrative in the YouTube comments dataset.
print("Top narrative in YouTube comments:")
print(comments_narrative_counts.sort_values("count", ascending=False).head(1))

Top narrative in news:
         narrative  count platform
2  monetary_causes     27     news

Top narrative in YouTube videos:
         narrative  count        platform
2  monetary_causes     19  youtube_videos

Top narrative in YouTube comments:
  narrative  count          platform
3     other    336  youtube_comments


In [41]:
import pandas as pd
import plotly.express as px

# Define a consistent order for platforms and narrative categories
# so the chart remains easy to read and compare.
platform_order = ["news", "youtube_videos", "youtube_comments"]
narrative_order = [
    "corporate_greed",
    "political_failure",
    "monetary_causes",
    "energy_tax_costs",
    "other",
]

# Create a copy of the combined count table to avoid changing
# the original summary data.
stacked_df = combined_counts.copy()

# Create all possible platform-narrative combinations.
# Missing combinations are later filled with zero so that every
# platform is shown with the same narrative structure.
full_index = pd.MultiIndex.from_product(
    [platform_order, narrative_order],
    names=["platform", "narrative"],
)

stacked_df = (
    stacked_df.set_index(["platform", "narrative"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Calculate the total number of texts for each platform.
# This is needed to convert absolute counts into percentages.
stacked_df["platform_total"] = (
    stacked_df.groupby("platform")["count"].transform("sum")
)

# Calculate the percentage share of each narrative within each platform.
# This allows a fair comparison despite different dataset sizes.
stacked_df["percentage"] = (
    stacked_df["count"] / stacked_df["platform_total"] * 100
)

# Map internal platform names to cleaner labels for the chart.
platform_labels = {
    "news": "News",
    "youtube_videos": "YouTube Videos",
    "youtube_comments": "YouTube Comments",
}
stacked_df["platform_label"] = stacked_df["platform"].map(platform_labels)

# Create an interactive stacked bar chart.
# Each bar represents one platform and is normalized through
# percentage shares within that platform.
fig = px.bar(
    stacked_df,
    x="platform_label",
    y="percentage",
    color="narrative",
    category_orders={
        "platform_label": ["News", "YouTube Videos", "YouTube Comments"],
        "narrative": narrative_order,
    },
    custom_data=["count", "percentage", "platform_total"],
    title="Narrative Distribution across Media Platforms (Stacked)",
)

# Customize the hover information so both counts and percentages
# are visible when moving the cursor over a bar segment.
fig.update_traces(
    hovertemplate=(
        "<b>Platform:</b> %{x}<br>"
        "<b>Narrative:</b> %{fullData.name}<br>"
        "<b>Count:</b> %{customdata[0]}<br>"
        "<b>Percentage:</b> %{customdata[1]:.1f}%<br>"
        "<b>Total texts on platform:</b> %{customdata[2]}<extra></extra>"
    )
)

# Adjust the layout for readability and ensure the y-axis
# always covers the full 0 to 100 percent range.
fig.update_layout(
    barmode="stack",
    xaxis_title="Platform",
    yaxis_title="Percentage within platform",
    yaxis=dict(range=[0, 100]),
    legend_title="Narrative",
    template="plotly_white",
)

# Open the interactive chart in the browser.
fig.show("browser")

In [42]:
import pandas as pd
import plotly.express as px

# Define a consistent order for platforms and narrative categories
# so the heatmap remains structured and easy to compare.
platform_order = ["news", "youtube_videos", "youtube_comments"]
narrative_order = [
    "corporate_greed",
    "political_failure",
    "monetary_causes",
    "energy_tax_costs",
    "other",
]

# Create a copy of the combined count table to avoid modifying
# the original summary data.
heatmap_df = combined_counts.copy()

# Create all possible platform-narrative combinations.
# Missing combinations are filled with zero so that every platform
# has the same set of narrative categories in the heatmap.
full_index = pd.MultiIndex.from_product(
    [platform_order, narrative_order],
    names=["platform", "narrative"],
)

heatmap_df = (
    heatmap_df.set_index(["platform", "narrative"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Calculate the total number of texts per platform.
# This is needed to convert raw counts into comparable percentages.
heatmap_df["platform_total"] = (
    heatmap_df.groupby("platform")["count"].transform("sum")
)

# Calculate the percentage share of each narrative within each platform.
heatmap_df["percentage"] = (
    heatmap_df["count"] / heatmap_df["platform_total"] * 100
)

# Replace internal platform names with cleaner labels for display.
platform_labels = {
    "news": "News",
    "youtube_videos": "YouTube Videos",
    "youtube_comments": "YouTube Comments",
}
heatmap_df["platform_label"] = heatmap_df["platform"].map(platform_labels)

# Create a matrix of percentage values for the heatmap.
# Rows represent platforms and columns represent narratives.
heatmap_matrix = heatmap_df.pivot(
    index="platform_label",
    columns="narrative",
    values="percentage",
)

# Reorder rows and columns so the heatmap follows a consistent structure.
heatmap_matrix = heatmap_matrix.reindex(
    index=["News", "YouTube Videos", "YouTube Comments"],
    columns=narrative_order,
)

# Create a second matrix with absolute counts.
# These values are used only for the hover information.
count_matrix = heatmap_df.pivot(
    index="platform_label",
    columns="narrative",
    values="count",
).reindex(
    index=["News", "YouTube Videos", "YouTube Comments"],
    columns=narrative_order,
)

# Create an interactive heatmap based on percentage values.
# Darker cells indicate stronger narrative presence within a platform.
fig = px.imshow(
    heatmap_matrix,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Blues",
    labels=dict(
        x="Narrative",
        y="Platform",
        color="Percentage",
    ),
    title="Narrative Intensity across Platforms (Heatmap)",
)

# Customize the hover display to show both percentage values
# and the corresponding absolute counts.
fig.update_traces(
    customdata=count_matrix.values,
    hovertemplate=(
        "<b>Platform:</b> %{y}<br>"
        "<b>Narrative:</b> %{x}<br>"
        "<b>Percentage:</b> %{z:.1f}%<br>"
        "<b>Count:</b> %{customdata}<extra></extra>"
    ),
)

# Apply a clean layout style.
fig.update_layout(
    template="plotly_white",
)

# Open the interactive heatmap in the browser.
fig.show("browser")

In [44]:
import pandas as pd
import plotly.express as px


# Use only the classified YouTube comments for this analysis.
comment_length_df = comments_classified.copy()

# Calculate the length of each comment in characters.
# This allows us to compare text length across narrative categories.
comment_length_df["comment_length"] = comment_length_df["comment"].str.len()

# Define a fixed narrative order so the box plot stays consistent
# with the other visualizations in the project.
narrative_order = [
    "corporate_greed",
    "political_failure",
    "monetary_causes",
    "energy_tax_costs",
    "other"
]

# Create an interactive box plot to compare the distribution
# of comment lengths across narrative categories.
fig = px.box(
    comment_length_df,
    x="narrative",
    y="comment_length",
    color="narrative",
    category_orders={"narrative": narrative_order},
    points="all",    #all #outliers
    title="Distribution of YouTube Comment Lengths by Narrative",
    labels={
        "narrative": "Narrative",
        "comment_length": "Comment length (characters)"
    },
    hover_data=["comment"]
)

# Apply a clean layout style and hide the legend because the colors
# already match the x-axis categories.
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

# Open the interactive box plot in the browser.
fig.show("browser")